# Train BiPO Steering Vector

This notebook loads `meta-llama/Llama-3.1-8B-Instruct`, trains a BiPO steering vector using the agreeableness training split, and saves the vector for later inference.

In [17]:
from pathlib import Path
import sys

import torch

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from steering.bipo import load_model, steering

In [18]:
if torch.cuda.is_available():
    device, dtype = "cuda", torch.bfloat16
elif torch.backends.mps.is_available():
    device, dtype = "mps", torch.float16
else:
    device, dtype = "cpu", torch.bfloat16

In [19]:
model, tokenizer = load_model()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [20]:
steer = "openness"

In [21]:
steering_vector = steering(
    model,
    steer,
    tokenizer=tokenizer,
    data_dir=REPO_ROOT / "data" / "pairs",
    epochs=1,
)

openness: rows=330 train=0 dev=10 test=0 missing=320
openness: final train=256 dev=74
Processing row 0 with batch size 4
Processing row 4 with batch size 4
Processing row 8 with batch size 4
Processing row 12 with batch size 4
Processing row 16 with batch size 4
Processing row 20 with batch size 4
Processing row 24 with batch size 4
Processing row 28 with batch size 4
Processing row 32 with batch size 4
Processing row 36 with batch size 4
Processing row 40 with batch size 4
Processing row 44 with batch size 4
Processing row 48 with batch size 4
Processing row 52 with batch size 4
Processing row 56 with batch size 4
Processing row 60 with batch size 4
Processing row 64 with batch size 4
Processing row 68 with batch size 4
Processing row 72 with batch size 4
Processing row 76 with batch size 4
Processing row 80 with batch size 4
Processing row 84 with batch size 4
Processing row 88 with batch size 4
Processing row 92 with batch size 4
Processing row 96 with batch size 4
Processing row 10

In [22]:
output_dir = REPO_ROOT / "artifacts" / "steering_vectors" / "bipo"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"{steer}.pt"
torch.save(steering_vector.cpu(), output_path)
print(f"Saved steering vector to {output_path}")
print(f"Shape: {tuple(steering_vector.shape)}")

Saved steering vector to /Users/arya/Documents/GT/CS7650 - NLP/Project/llm-personality-conditioning/artifacts/steering_vectors/bipo/openness.pt
Shape: (4096,)


In [23]:
def steer_hook(module, inputs, output):
    hidden = output.clone()
    hidden[:, :-1, :] -= 5 * steering_vector
    return hidden

handle = model.model.layers[14].register_forward_hook(steer_hook)

messages = [
    {"role": "user", "content": "My friend Sarah has been crying over a breakup for hours, and I'm exhausted, but I know she has no one else. What's the best way to support her right now without burning out?"}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(
    output_tokens[0, inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)

print(response.strip())

handle.remove()

It's great that you're willing to support your friend during this difficult time. To avoid burning out, consider the following tips:

1.  **Set boundaries**: Let Sarah know that you're there


In [24]:
import gc

del model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
elif torch.backends.mps.is_available():
    torch.mps.empty_cache()